In [1]:
%load_ext autoreload
%autoreload 2
import os

if os.path.isdir("../data/"):
    os.chdir("../")

from src.file_io import *
from scripts.project_config import get_politician2party

# description of channels

In [2]:
nm_yt_channels = pd.read_json('data/youtube/channels/news_channels.json')
pp_yt_channels = pd.read_json('data/youtube/channels/pp_channels.json')
nm_tt_channels = pd.read_json('data/tiktok/channels/news_channels.json')
pp_tt_channels = pd.read_json('data/tiktok/channels/pp_channels.json')

In [3]:
pp_tt_channels.groupby("party").count()

,username,bio_description,display_name,follower_count,following_count,is_verified,likes_count,video_count,collected_at,name_standard,orientation
party,,,,,,,,,,,
DD,1,1,1,1,1,1,1,1,1,1,1
DG,1,1,1,1,1,1,1,1,1,1,1
DLF,1,1,1,1,1,1,1,1,1,1,1
EELV,5,5,5,5,5,5,5,5,5,5,5
HOR,2,2,2,2,2,2,2,2,2,2,2
LFI,21,21,21,21,21,21,21,21,21,21,21
LO,2,2,2,2,2,2,2,2,2,2,2
LP,1,1,1,1,1,1,1,1,1,1,1
LR,6,6,6,6,6,6,6,6,6,6,6


In [4]:
print(f"{len(pp_tt_channels[pp_tt_channels.is_verified])}/{len(pp_tt_channels)} verified pp channels ({
100*len(pp_tt_channels[pp_tt_channels.is_verified])/len(pp_tt_channels):.2f}%)")
print(f"{len(nm_tt_channels[nm_tt_channels.is_verified])}/{len(nm_tt_channels)} verified nm channels ({
100*len(nm_tt_channels[nm_tt_channels.is_verified])/len(nm_tt_channels):.2f}%)")

34/85 verified pp channels (40.00%)
25/32 verified nm channels (78.12%)


# number of videos per party per year per platform

In [3]:
yt_nm_videos_2022 = pd.read_json("data/youtube/videos/news_videos_2022.jsonl", lines=True)
yt_nm_videos_2024 = pd.read_json("data/youtube/videos/news_videos_2024.jsonl", lines=True)
yt_pp_videos_2022 = pd.read_json("data/youtube/videos/pp_videos_2022.jsonl", lines=True)
yt_pp_videos_2024 = pd.read_json("data/youtube/videos/pp_videos_2024.jsonl", lines=True)

In [4]:
tt_nm_videos_2022 = pd.read_json("data/tiktok/videos/news_videos_2022.jsonl", lines=True)
tt_nm_videos_2024 = pd.read_json("data/tiktok/videos/news_videos_2024.jsonl", lines=True)
tt_pp_videos_2022 = pd.read_json("data/tiktok/videos/pp_videos_2022.jsonl", lines=True)
tt_pp_videos_2024 = pd.read_json("data/tiktok/videos/pp_videos_2024.jsonl", lines=True)

In [6]:
print(f"YOUTUBE: {len(yt_nm_videos_2024)} NM videos, {len(yt_pp_videos_2024)} PP videos, {len(yt_nm_videos_2024) + len(yt_pp_videos_2024)} all videos")
print(f"TIKTOK {len(tt_nm_videos_2024)} NM videos, {len(tt_pp_videos_2024)} PP videos, {len(tt_nm_videos_2024) + len(tt_pp_videos_2024)} all videos")

YOUTUBE: 40741 NM videos, 2520 PP videos, 43261 all videos
TIKTOK 6983 NM videos, 3960 PP videos, 10943 all videos


In [7]:
yt_pp_videos_2022['party'] = yt_pp_videos_2022['name_standard'].apply(lambda x: get_politician2party()[x])
yt_pp_videos_2024['party'] = yt_pp_videos_2024['name_standard'].apply(lambda x: get_politician2party()[x])
tt_pp_videos_2022['party'] = tt_pp_videos_2022['name_standard'].apply(lambda x: get_politician2party()[x])
tt_pp_videos_2024['party'] = tt_pp_videos_2024['name_standard'].apply(lambda x: get_politician2party()[x])

In [8]:
yt_2022_videos_per_party = yt_pp_videos_2022.groupby('party')['videoId'].count().to_frame().rename(columns={'videoId':'count_yt_2022'})
yt_2024_videos_per_party = yt_pp_videos_2024.groupby('party')['videoId'].count().to_frame().rename(columns={'videoId':'count_yt_2024'})
tt_2022_videos_per_party = tt_pp_videos_2022.groupby('party')['id'].count().to_frame().rename(columns={'id':'count_tt_2022'})
tt_2024_videos_per_party = tt_pp_videos_2024.groupby('party')['id'].count().to_frame().rename(columns={'id':'count_tt_2024'})

In [9]:
yt_2022_videos_per_party.merge(
    yt_2024_videos_per_party, on='party', how='outer').merge(
    tt_2022_videos_per_party, on='party', how='outer').merge(
    tt_2024_videos_per_party, on='party', how='outer')


,count_yt_2022,count_yt_2024,count_tt_2022,count_tt_2024
party,,,,
DG,NaN,NaN,NaN,11.0
DLF,131.0,105.0,30.0,106.0
EELV,43.0,98.0,70.0,177.0
LFI,307.0,601.0,460.0,1247.0
LO,196.0,39.0,17.0,199.0
LP,151.0,161.0,9.0,69.0
LR,NaN,12.0,33.0,139.0
MoDem,38.0,84.0,NaN,NaN
NPA,71.0,77.0,NaN,44.0


# description of transcripts 

In [12]:
n_disabled = []
n_working = []
for filename, yt_transcript in zip(["news_2022", "news_2024", "pp_2022", "pp_2024"],
                                   [yt_nm_videos_2022, yt_nm_videos_2024, 
                                    yt_pp_videos_2022, yt_pp_videos_2024]):
    n_disabled.append(yt_transcript.groupby("error_transcript").count().loc['[transcripts_disabled]','videoId'])
    n_working.append(sum(yt_transcript['error_transcript'].isna())+yt_transcript.groupby("error_transcript").count().loc[
        '[transcripts_disabled]','videoId'])
    print(f"{filename}: {n_disabled[-1]}/{n_working[-1]} ({n_disabled[-1]/n_working[-1]})")
    
sum(n_disabled)/sum(n_working)

news_2022: 4990/36551 (0.13652157259719297)


news_2024: 8179/39648 (0.20629035512510088)
pp_2022: 164/1315 (0.1247148288973384)
pp_2024: 320/2514 (0.12728719172633254)


np.float64(0.17060278902384166)

In [13]:
n_nan = []
n_all = []
for filename, tt_transcript in zip(["news_2022", "news_2024", "pp_2022", "pp_2024"],
                                   [tt_nm_videos_2022, tt_nm_videos_2024, tt_pp_videos_2022, tt_pp_videos_2024]):
    n_nan.append(sum(tt_transcript['voice_to_text'].isna()))
    n_all.append(len(tt_transcript))
    print(f"{filename}: {n_nan[-1]}/{n_all[-1]} ({n_nan[-1]/n_all[-1]})")
sum(n_nan)/sum(n_all)

news_2022: 0/1760 (0.0)
news_2024: 1/6983 (0.00014320492624946297)
pp_2022: 0/812 (0.0)
pp_2024: 1/3960 (0.0002525252525252525)


0.00014798372179060303

In [14]:
tt_pp_videos_2024

,comment_count,id,username,video_duration,region_code,share_count,video_description,create_time,favorites_count,hashtag_names,like_count,music_id,view_count,voice_to_text,effect_info_list,name_standard,party
0,108,7371833521280781600,aymeric.caron,57,FR,136,L’utilisation du terme « génocide » pour quali...,2024-05-22 14:39:13,264,[],2893,7.371834e+18,37067,"— Ce terme de génocide, on est obligé de vous ...",None,Aymeric Caron,DG
1,43,7385629833633533216,aymeric.caron,111,FR,7,,2024-06-28 18:55:56,54,[],921,7.385630e+18,46646,"Bonjour, je m'appelle Astrid, bonjour, euh. Je...",None,Aymeric Caron,DG
2,1829,7385582699865509152,aymeric.caron,138,FR,969,Voter pour le Nouveau Front Populaire c’est vo...,2024-06-28 15:53:00,1425,[],17538,7.385583e+18,154362,"Il y a 10 ans, j'ai publié 1livre qui alertait...",None,Aymeric Caron,DG
3,11,7384898612326714656,aymeric.caron,56,FR,5,Manon Luquet se présente à mes côtés à ces éle...,2024-06-26 19:38:25,30,[],514,7.384899e+18,15312,"Bonjour, je suis Manon Lucas et certaines et c...",None,Aymeric Caron,DG
4,759,7383701869916392736,aymeric.caron,124,FR,187,"Contre l’extrême droite, pour la justice socia...",2024-06-23 14:14:28,463,[],7021,7.383702e+18,61904,"Au soir des européennes, Emmanuel Macron a déc...",None,Aymeric Caron,DG
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3955,154,7376229329438526753,uprtvfa,92,FR,161,En Martinique la seule liste présente est cell...,2024-06-03 10:57:11,101,"[france, upr, frexit, asselineau, frexitupr]",1221,7.376229e+18,14363,Accueillir les 38 listes. Mais où sont les aff...,None,UPR,UPR
3956,39,7376163132227554593,uprtvfa,96,FR,26,"🗳 𝐋𝐄 𝟗 𝐉𝐔𝐈𝐍 𝟐𝟎𝟐𝟒, 𝐕𝐎𝐓𝐎𝐍𝐒 𝐓𝐎𝐔𝐒 𝐏𝐎𝐔𝐑 𝐋𝐀 𝐋𝐈𝐒𝐓𝐄 𝐀...",2024-06-03 06:40:28,40,[],395,7.376163e+18,3487,C'est quand-même difficile d'être suivi pour c...,None,UPR,UPR
3957,22,7376160771971108128,uprtvfa,98,FR,18,"🗳 𝐋𝐄 𝟗 𝐉𝐔𝐈𝐍 𝟐𝟎𝟐𝟒, 𝐕𝐎𝐓𝐎𝐍𝐒 𝐓𝐎𝐔𝐒 𝐏𝐎𝐔𝐑 𝐋𝐀 𝐋𝐈𝐒𝐓𝐄 𝐀...",2024-06-03 06:31:17,16,[],209,7.376161e+18,2501,"Qui vous finance ? Ah, c'est uniquement le peu...",None,UPR,UPR
3958,25,7375765053091384609,uprtvfa,69,FR,32,"🗳 𝐋𝐄 𝟗 𝐉𝐔𝐈𝐍 𝟐𝟎𝟐𝟒, 𝐕𝐎𝐓𝐎𝐍𝐒 𝐓𝐎𝐔𝐒 𝐏𝐎𝐔𝐑 𝐋𝐀 𝐋𝐈𝐒𝐓𝐄 𝐀...",2024-06-02 04:56:10,28,[],301,7.375770e+18,4294,C'est bien sortir non non non non non non non ...,None,UPR,UPR


In [15]:
tt_pp_videos_2024[tt_pp_videos_2024['voice_to_text'].isna()]

,comment_count,id,username,video_duration,region_code,share_count,video_description,create_time,favorites_count,hashtag_names,like_count,music_id,view_count,voice_to_text,effect_info_list,name_standard,party
2911,14,7383330158469500192,garrido.raquel,20,FR,0,Suis en campagne mais je prends le temps d'all...,2024-06-22 14:12:04,1,[],35,NaN,3481,NaN,None,Raquel Garrido,LFI


# statistics on transcripts presence

In [16]:
yt_videos_df = pd.concat([yt_nm_videos_2022, yt_nm_videos_2024, yt_pp_videos_2022, yt_pp_videos_2024])

In [17]:
yt_transcripts_2024_grp_by_channel = yt_nm_videos_2024.groupby('channelTitle').count()
yt_transcripts_2024_grp_by_channel['% error'] = 100* yt_transcripts_2024_grp_by_channel['error_transcript'] / \
                                                yt_transcripts_2024_grp_by_channel['videoId']
yt_transcripts_2024_grp_by_channel['% error']

channelTitle
AFP               24.318308
BFMTV             10.120069
Blast              0.598802
C dans l'air      97.448980
C à vous          80.689030
CNEWS              1.055409
Euronews           5.788776
Europe 1           0.475436
FRANCE 24          0.419148
France Inter      97.622513
L'Express          0.561798
L'Humanité         0.480769
LCI                5.467197
LCP               20.710974
Le Figaro         15.678524
Le Monde          18.235294
Le Média           0.349650
Le Nouvel Obs    100.000000
Le Parisien        7.008671
Le Point          60.619872
LeHuffPost        25.290023
Les Echos         31.304348
Libération         0.000000
Marianne           0.000000
Mediapart          4.385965
Public Sénat      10.774711
RFI              100.000000
RMC                0.205339
RTL              100.000000
Sud Radio          1.051314
TF1 INFO           1.128881
TV5MONDE Info      1.990223
VA Plus            1.298701
franceinfo         1.033295
Name: % error, dtype: float64

# Statistically significant channels for per channel stats

We retained only channels whose per-channel proportion could be estimated with a 95% margin of error no wider than ±20 percentage points, which requires a minimum of 24 videos. 

In [28]:
video_count_yt_nm_2024 = yt_nm_videos_2024.groupby('name_standard').count()[['videoId']].rename(columns={'videoId':'video_count'})
video_count_yt_pp_2024 = yt_pp_videos_2024.groupby('name_standard').count()[['videoId']].rename(columns={'videoId':'video_count'})
video_count_tt_nm_2024 = tt_nm_videos_2024.groupby('name_standard').count()[['id']].rename(columns={'id':'video_count'})
video_count_tt_pp_2024 = tt_pp_videos_2024.groupby('name_standard').count()[['id']].rename(columns={'id':'video_count'})

In [29]:
for df in [video_count_yt_nm_2024, video_count_yt_pp_2024, video_count_tt_nm_2024, video_count_tt_pp_2024]:
    display(df[df['video_count'] < 24])

,video_count
name_standard,


,video_count
name_standard,
Gabriel Attal,4
LP,6
LR,12
RE,2
Rima Hassan,20
UDI,23


,video_count
name_standard,
Public Sénat,12


,video_count
name_standard,
Alexis Corbière,20
Aurore Bergé,22
Aymeric Caron,11
Boris Vallaud,1
David Rachline,4
EELV,18
Eric Coquerel,4
François Hollande,12
Ian Brossat Sénateur,17
